# 🧪 Phase 1: Baseline Proof of Concept (POC)
**Project:** Multi-PDF RAG Engine Baseline Feasibility
**Date:** August 2026 | **Author:** Sanket Kakad

> **POC Note:** This notebook documents the initial baseline experimentation and Proof of Concept (POC) executed prior to engineering the production architecture (FastAPI + Groq Llama 3.1 8B Instant + PyMuPDF Page-Aware Chunking + BM25 Hybrid + FlashRank Reranking).

---
### POC Milestones Accomplished:
1. ✅ **Text Extraction Validation**: Extracted text blocks from PDF documents using PyMuPDF ().
2. ✅ **Text Chunking Validation**: Split documents using  (1000 char chunks, 200 overlap).
3. ✅ **Vector Storage Prototype**: Embedded text using Google/OpenAI embeddings and stored vectors in local ChromaDB.
4. ✅ **Prompt Engineering & Validation**: Executed context-constrained LLM prompting to prevent hallucination.
5. ✅ **Interactive QA Loop**: Verified similarity search retrieval over sample papers (e.g. *Attention Is All You Need*).


In [16]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file


True

In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0.2, max_output_tokens=1024)

In [18]:
import fitz

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text



In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_text_into_chunks(text, chunk_size=1000, overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )
    return text_splitter.split_text(text)


In [20]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)


In [21]:
from langchain_chroma import Chroma

def store_chunks_in_chroma(split_chunks):
    db = Chroma.from_texts(
        texts=split_chunks,
        embedding=embeddings,
        persist_directory="./chroma_db"   
    )

    return db

In [22]:
def load_vector_db():
    return Chroma(
        persist_directory="./chroma_db",
        embedding_function=embeddings
    )

In [ ]:
def ask_question(db, question, k=3):
    # Step 1: retrieve relevant chunks
    docs = db.similarity_search(question, k=k)

    context = "\n\n".join([d.page_content for d in docs])

    # Step 2: build prompt
    prompt = f"""
You are a precise and reliable assistant for answering questions based on provided context.

RULES:
- Use ONLY the information in the context below.
- Do NOT use any external knowledge or assumptions.
- If the answer is not fully contained in the context, say: "I don't know based on the provided context."
- Be concise and accurate.
- If multiple pieces of context are relevant, combine them logically.
- Do not repeat the context unless necessary.
- Always prioritize correctness over completeness.

Context:
{context}

Question:
{question}

Answer:
"""
    
    print("Prompt sent to Gemini:",prompt)  # Debugging: print the prompt to see what is being sent to Gemini

    # Step 3: get response from Gemini
    response = llm.invoke(prompt)

    return response.content

In [24]:
pdf_path = "../documents/NIPS-2017-attention-is-all-you-need-Paper.pdf"
pdf_text = extract_text_from_pdf(pdf_path)
print(len(pdf_text))

split_chunks = split_text_into_chunks(pdf_text)
print(f"Number of chunks: {len(split_chunks)}") 
db = store_chunks_in_chroma(split_chunks)

print("Vector DB created and saved locally")

32709
Number of chunks: 41
Vector DB created and saved locally


In [25]:
db = load_vector_db()

while True:
    query = input("\nAsk a question (or 'exit'): ")

    if query.lower() == "exit":
        break

    answer = ask_question(db, query)

    print("\n🤖 Answer:\n", answer)

Prompt sent to Gemini: 
You are a helpful assistant.
Answer ONLY using the context below.

Context:
In this work we propose the Transformer, a model architecture eschewing recurrence and instead
relying entirely on an attention mechanism to draw global dependencies between input and output.
The Transformer allows for signiﬁcantly more parallelization and can reach a new state of the art in
translation quality after being trained for as little as twelve hours on eight P100 GPUs.
2
Background
The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU
[20], ByteNet [15] and ConvS2S [8], all of which use convolutional neural networks as basic building
block, computing hidden representations in parallel for all input and output positions. In these models,
the number of operations required to relate signals from two arbitrary input or output positions grows
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This make

---
## 🎯 POC Conclusion & Production Transition Plan
Following the successful validation of this baseline POC, the production engine was upgraded to:
- **Groq Llama 3.1 8B Instant**: Sub-500ms LLM generation.
- **Page-Aware Ingestion**: Page-indexed metadata (, , ) for exact inline citations.
- **Dense + BM25 Hybrid Retrieval**: Combining semantic vector search with keyword matching via Reciprocal Rank Fusion (RRF).
- **FlashRank Re-ranking**: Cross-encoder passage scoring.
- **Streamlit & FastAPI REST UI**: Interactive ChatGPT-like interface & serverless Vercel endpoints.
